# Downloading the Data
Bring in the data in order to run it against the Decision Tree and LSTM

In [1]:
import kagglehub
import tensorflow as tf
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
import pandas as pd
from sklearn.model_selection import StratifiedKFold

from sklearn.metrics import precision_recall_fscore_support, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

from datasets import load_dataset

ds = load_dataset("zh-plus/tiny-imagenet")

y_all = np.array(ds["train"]["label"])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=100)

/Users/armindelmo/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/armindelmo/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

## Decision Trees
Using a forest to properly be able to classify these images

In [2]:
# getting the dataset ready
import tensorflow_decision_forests as tfdf

def hf_to_tf(hf_split, batch_size = 64, image_size = (64,64), shuffle=False):
    def gen():
        for example in hf_split:
            img = example["image"].convert("RGB").resize(image_size)
            arr = np.asarray(img, dtype=np.float32) / 255.0
            yield arr, np.int32(example["label"])

    output_signature = (
        tf.TensorSpec(shape=(image_size[0], image_size[1], 3), dtype=tf.float32),
        tf.TensorSpec(shape=(), dtype=tf.int32),
    )

    ds_tf = tf.data.Dataset.from_generator(gen, output_signature=output_signature)
    if shuffle:
        ds_tf = ds_tf.shuffle(10000, reshuffle_each_iteration=True)
    ds_tf = ds_tf.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds_tf

train_ds = hf_to_tf(ds["train"], batch_size=64, image_size=(64, 64), shuffle=True)
test_ds = hf_to_tf(ds["valid"], batch_size=64, image_size=(64, 64), shuffle=False)


2026-05-03 22:28:26.452795: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Pro
2026-05-03 22:28:26.452826: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-05-03 22:28:26.452850: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-05-03 22:28:26.453043: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:306] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-03 22:28:26.453346: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:272] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


In [7]:
X_val_list, y_val_list = [], []
X_train_list, y_train_list = [], []
pca = PCA(n_components=806, svd_solver="randomized", random_state=100)

for imgs, labels in test_ds:
    flat = tf.reshape(imgs, [imgs.shape[0], -1])
    X_val_list.append(flat.numpy())
    y_val_list.append(labels.numpy())

X_test_val = np.concatenate(X_val_list, axis=0)
y_test_val = np.concatenate(y_val_list, axis=0)

# pca
X_train = np.concatenate(X_train_list, axis=0)
pca.fit(X_train)

X_test_pca = pca.transform(X_test_val)

feature_names = [f"p{i}" for i in range(X_test_pca.shape[1])]
testing_df = pd.DataFrame(X_test_pca, columns=feature_names)
testing_df["label"] = y_test_val.astype(int)

testing_tfds = tfdf.keras.pd_dataframe_to_tf_dataset(testing_df, label="label", max_num_classes=200)

KeyboardInterrupt: 

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

fold_acc = []
fold_f1, fold_p, fold_r = [], [], []

test_acc = []
test_f1, test_p, test_r = [], [], []

for fold, (train_index, valid_index) in enumerate(skf.split(np.zeros(len(y_all)), y_all), 1):
    print(f"\n-=-=--=-=Fold {fold}=-=-=-=-")
    train_fold = ds["train"].select(train_index.tolist())
    valid_fold = ds["train"].select(valid_index.tolist())

    train_ds_fold = hf_to_tf(train_fold, shuffle=True)
    valid_ds_fold = hf_to_tf(valid_fold, shuffle=False)

    X_tlist, y_tlist = [], []

    # flatten training data to fit in a decision tree
    for imgs, labels in train_ds_fold:
        flat = tf.reshape(imgs, [tf.shape(imgs)[0], -1])
        X_tlist.append(flat.numpy())
        y_tlist.append(labels.numpy())

    X_train = np.concatenate(X_tlist, axis=0)
    y_train = np.concatenate(y_tlist, axis=0)

    # flatten validation data to fit in a decision tree
    X_vlist, y_vlist = [], []

    # flatten training data to fit in a decision tree
    for imgs, labels in valid_ds_fold:
        flat = tf.reshape(imgs, [tf.shape(imgs)[0], -1])
        X_vlist.append(flat.numpy())
        y_vlist.append(labels.numpy())

    X_valid = np.concatenate(X_vlist, axis=0)
    y_valid = np.concatenate(y_vlist, axis=0)

    # reduce features
    pca = PCA(n_components=806, svd_solver="randomized", random_state=100)
    X_train_pca = pca.fit_transform(X_train)
    X_valid_pca = pca.transform(X_valid)

    # build dataframe for features
    feature_names = [f"p{i}" for i in range(X_train_pca.shape[1])]

    train_df = pd.DataFrame(X_train_pca, columns=feature_names)
    train_df["label"] = y_train.astype(int)

    valid_df = pd.DataFrame(X_valid_pca, columns=feature_names)
    valid_df["label"] = y_valid.astype(int)

    train_tfds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df, label="label", max_num_classes=200)
    valid_tfds = tfdf.keras.pd_dataframe_to_tf_dataset(valid_df, label="label", max_num_classes=200)

    # train df
    model = tfdf.keras.RandomForestModel(num_trees=64, max_depth=16)
    model.fit(train_tfds)

    # evaluate, grab information
    model.compile(metrics=["accuracy"])
    res = model.evaluate(valid_tfds, return_dict=True, verbose = 0)
    fold_acc.append(res["accuracy"])

    y_prob = model.predict(valid_tfds, verbose = 0)
    y_pred = np.argmax(y_prob, axis=1)

    # Aggregate metrics
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_valid, y_pred, average="weighted", zero_division=0
    )
    fold_f1.append(f1_weighted)
    fold_p.append(p_weighted)
    fold_r.append(r_weighted)
    print("val_acc:", res["accuracy"], 
          "val_f1_weighted:", f1_weighted,
          "val_p_weighted:", p_weighted,
          "val_r_weighted:", r_weighted
          )
    X_test_list, y_test_list = [], []
    for imgs, labels in test_ds:
        flat = tf.reshape(imgs, [imgs.shape[0], -1])
        X_test_list.append(flat.numpy())
        y_test_list.append(labels.numpy())

    X_test_val = np.concatenate(X_test_list, axis=0)
    y_test_val = np.concatenate(y_test_list, axis=0)

    X_test_pca = pca.transform(X_test_val)

    feature_names = [f"p{i}" for i in range(X_test_pca.shape[1])]
    testing_df = pd.DataFrame(X_test_pca, columns=feature_names)
    testing_df["label"] = y_test_val.astype(int)

    testing_tfds = tfdf.keras.pd_dataframe_to_tf_dataset(testing_df, label="label", max_num_classes=200)
    test_results = model.evaluate(testing_tfds, return_dict=True, verbose=0)
    test_acc.append(test_results["accuracy"])
    # collecting results from the testing set
    val_results = model.evaluate(testing_tfds.prefetch(tf.data.AUTOTUNE))

    # cooked or cooking?
    y_true = np.concatenate([y.numpy() for _, y in testing_tfds], axis=0)

    y_prob = model.predict(testing_tfds.prefetch(tf.data.AUTOTUNE), verbose=1)
    y_pred = np.argmax(y_prob, axis=1)

    # calculate metrics
    pt_weighted, rt_weighted, f1t_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    test_p.append(pt_weighted)
    test_r.append(rt_weighted)
    test_f1.append(f1t_weighted)
    
print("\nCV accuracy mean/std:", np.mean(fold_acc), np.std(fold_acc))
print("CV F1(w) mean/std   :", np.mean(fold_f1), np.std(fold_f1))



-=-=--=-=Fold 1=-=-=-=-


KeyboardInterrupt: 

In [69]:
# checking the accuracy on the training set
train_results = model.evaluate(train_tfds, return_dict=True)
print(train_results)


100/100 [==============================] - 11s 108ms/step - loss: 0.0000e+00 - accuracy: 0.7765
{'loss': 0.0, 'accuracy': 0.7764700055122375}


# Collecting Statistics

Now that we have ran all of the models, we are going to collect their statistics with scikit-learn.
We can accomplish this by collecting their accuracy from the validation set, and go from there.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support, ConfusionMatrixDisplay, confusion_matrix
import matplotlib.pyplot as plt

# collecting results from the lstm
val_results = model.evaluate(testing_tfds.prefetch(tf.data.AUTOTUNE))

# using scikit-learn stuff
y_true = np.concatenate([y.numpy() for _, y in testing_tfds], axis=0)

y_prob = model.predict(testing_tfds.prefetch(tf.data.AUTOTUNE), verbose=1)
y_pred = np.argmax(y_prob, axis=1)

# get metrics out
p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

cm = confusion_matrix(y_true, y_pred)

disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=range(200))
disp.plot(cmap=plt.cm.Blues)
plt.title("Decision Forest Confusion Matrix")
plt.show()

print("Testing >> Weighted- Precision:", p_weighted, "Recall:", r_weighted, "F1:", f1_weighted)

KeyError: 0